Step 1: Keep only the price event in market dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/data cleaning"
titles_path = f"{base_path}/polymarket_stock_price_titles.xlsx"
markets_path = f"{base_path}/stocks_markets.csv"
output_path = f"{base_path}/stocks_markets_matched.csv"

titles_df = pd.read_excel(titles_path)
markets_df = pd.read_csv(markets_path)

titles_df["id"] = titles_df["id"].astype(str).str.strip()
markets_df["event_id"] = markets_df["event_id"].astype(str).str.strip()

matched_df = markets_df[markets_df["event_id"].isin(titles_df["id"])].copy()
matched_df.to_csv(output_path, index=False)

print(f"saved to: {output_path}")
print(f"rows kept: {len(matched_df)}")

saved to: /content/drive/MyDrive/data cleaning/stocks_markets_matched.csv
rows kept: 6678


正则表达式提取法

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

# ========= 1. 文件路径 =========
base_path = "/content/drive/MyDrive/data cleaning"
input_path = f"{base_path}/stocks_markets_matched.csv"
output_path = f"{base_path}/stocks_markets_with_bounds.csv"

# ========= 2. 读取文件 =========
df = pd.read_csv(input_path)

# 如果有 question 列就直接用；否则按 B 列取第二列
if "question" in df.columns:
    question_col = "question"
else:
    question_col = df.columns[1]

print("Using question column:", question_col)
print("Total rows:", len(df))

# ========= 3. 解析函数 =========
def parse_price_bounds(question):
    if pd.isna(question):
        return np.nan, np.nan

    q = str(question).strip()
    q_lower = q.lower()
    q_clean = q.replace(",", "")

    # 1) 区间: $100-$105 / between $100 and $105 / $100 to $105
    range_patterns = [
        r"\$?\s*(\d+(?:\.\d+)?)\s*[-–]\s*\$?\s*(\d+(?:\.\d+)?)",
        r"between\s+\$?\s*(\d+(?:\.\d+)?)\s+and\s+\$?\s*(\d+(?:\.\d+)?)",
        r"\$?\s*(\d+(?:\.\d+)?)\s+to\s+\$?\s*(\d+(?:\.\d+)?)",
    ]
    for pat in range_patterns:
        m = re.search(pat, q_clean, flags=re.IGNORECASE)
        if m:
            a = float(m.group(1))
            b = float(m.group(2))
            return min(a, b), max(a, b)

    # 2) above / over / greater than / higher than / more than
    m = re.search(
        r"(?:above|over|greater than|higher than|more than)\s+\$?\s*(\d+(?:\.\d+)?)",
        q_clean,
        flags=re.IGNORECASE,
    )
    if m:
        return float(m.group(1)), np.inf

    # 3) below / under / less than / lower than
    m = re.search(
        r"(?:below|under|less than|lower than)\s+\$?\s*(\d+(?:\.\d+)?)",
        q_clean,
        flags=re.IGNORECASE,
    )
    if m:
        return -np.inf, float(m.group(1))

    # 4) reach / hit
    m = re.search(
        r"(?:reach|hit)\s+\$?\s*(\d+(?:\.\d+)?)",
        q_clean,
        flags=re.IGNORECASE,
    )
    if m:
        return float(m.group(1)), np.inf

    # 5) close at / closes at / finish at / finishes at
    m = re.search(
        r"(?:close at|closes at|finish at|finishes at)\s+\$?\s*(\d+(?:\.\d+)?)",
        q_clean,
        flags=re.IGNORECASE,
    )
    if m:
        x = float(m.group(1))
        return x, x

    return np.nan, np.nan

# ========= 4. 批量解析 =========
parsed = df[question_col].apply(parse_price_bounds)
df[["lower_bound", "upper_bound"]] = pd.DataFrame(parsed.tolist(), index=df.index)

# 可选：把无穷改成字符串，方便看
df["lower_bound"] = df["lower_bound"].replace({-np.inf: "-inf", np.inf: "inf"})
df["upper_bound"] = df["upper_bound"].replace({-np.inf: "-inf", np.inf: "inf"})

# ========= 5. 输出 =========
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print("Failed rows:", df["lower_bound"].isna().sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using question column: question
Total rows: 6678
Saved to: /content/drive/MyDrive/data cleaning/stocks_markets_with_bounds.csv
Failed rows: 669


全部行都用LLM提取

In [ ]:
!pip install pandas requests tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm

OPENROUTER_API_KEY = "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"
MODEL = "openrouter/elephant-alpha"

base_path = "/content/drive/MyDrive/data cleaning"
input_path = f"{base_path}/stocks_markets_matched.csv"
output_path = f"{base_path}/stocks_markets_llm_parsed_all.csv"

BATCH_SIZE = 200
SLEEP_BETWEEN_CALLS = 1.0
MAX_RETRIES = 5
N_ROWS = 6679

df = pd.read_csv(input_path)

if "question" in df.columns:
    question_col = "question"
else:
    question_col = df.columns[1]

df = df.head(N_ROWS).copy().reset_index(drop=False).rename(columns={"index": "row_id"})

print("Using question column:", question_col)
print("Trial rows:", len(df))


def chunk_list(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]


def build_messages(batch_records):
    system_prompt = """
You are an information extraction system.

For each input row, extract:
1. ticker
2. lower_bound
3. upper_bound

Return ONLY valid JSON in this exact format:
{"results":[
  {
    "row_id": integer,
    "ticker": string or null,
    "lower_bound": number or "inf" or "-inf" or null,
    "upper_bound": number or "inf" or "-inf" or null
  }
]}

Rules:
- "reach $316", "hit $316", "touch $316", "dip to $316", "drop to $316", "fall to $316", "above $316", "over $316"
  => lower_bound = 316, upper_bound = "inf"
- "below $316", "under $316"
  => lower_bound = "-inf", upper_bound = 316
- "$100-$105" or "$100 to $105"
  => lower_bound = 100, upper_bound = 105
- exact bucket like "close at $100"
  => lower_bound = 100, upper_bound = 100
- Do not use date numbers as prices.
- Ignore years, dates, and week ranges unless they are clearly part of a price expression.
- If ticker appears in parentheses, use it.
- If ticker is not in parentheses but the company is obvious, infer the standard stock ticker.
- If not confident, return null for ticker.
- Return exactly one object per input row.

Examples:
"Will Apple reach $316 in November?" -> {"row_id":1,"ticker":"AAPL","lower_bound":316,"upper_bound":"inf"}
"Will Microsoft (MSFT) finish week of November 17 above $490?" -> {"row_id":2,"ticker":"MSFT","lower_bound":490,"upper_bound":"inf"}
"Will Netflix (NFLX) close at $100-$105 in 2025?" -> {"row_id":3,"ticker":"NFLX","lower_bound":100,"upper_bound":105}
"Will Apple dip to $268 in Nov 25-28?" -> {"row_id":4,"ticker":"AAPL","lower_bound":"-inf","upper_bound":"268"}
""".strip()

    user_prompt = "Extract fields for these rows:\n" + json.dumps(batch_records, ensure_ascii=False)

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_openrouter(batch_records):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL,
        "messages": build_messages(batch_records),
        "temperature": 0,
        "response_format": {"type": "json_object"},
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=180)
            response.raise_for_status()
            data = response.json()
            content = data["choices"][0]["message"]["content"]
            parsed = json.loads(content)

            if "results" not in parsed or not isinstance(parsed["results"], list):
                raise ValueError("Model output missing results array")

            return parsed["results"]

        except Exception as e:
            print(f"Retry {attempt+1}/{MAX_RETRIES} failed: {e}")
            time.sleep(2 * (attempt + 1))

    raise RuntimeError("OpenRouter call failed after retries")


records = (
    df[["row_id", question_col]]
    .rename(columns={question_col: "question"})
    .to_dict("records")
)

all_results = []

for batch in tqdm(list(chunk_list(records, BATCH_SIZE))):
    batch_results = call_openrouter(batch)
    all_results.extend(batch_results)
    time.sleep(SLEEP_BETWEEN_CALLS)

parsed_df = pd.DataFrame(all_results)

for col in ["row_id", "ticker", "lower_bound", "upper_bound"]:
    if col not in parsed_df.columns:
        parsed_df[col] = None

parsed_df = parsed_df.drop_duplicates(subset=["row_id"], keep="first")

final_df = df.merge(
    parsed_df[["row_id", "ticker", "lower_bound", "upper_bound"]],
    on="row_id",
    how="left"
)

final_df = final_df.drop(columns=["row_id"])
final_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(final_df[[question_col, "ticker", "lower_bound", "upper_bound"]].head(30))
print("Rows with both bounds missing:", ((final_df["lower_bound"].isna()) & (final_df["upper_bound"].isna())).sum())

Using question column: question
Trial rows: 6678


 38%|███▊      | 13/34 [07:43<11:02, 31.54s/it]

Retry 1/5 failed: Unterminated string starting at: line 1 column 80147 (char 80146)
Retry 2/5 failed: Expecting ',' delimiter: line 1 column 78258 (char 78257)


 79%|███████▉  | 27/34 [20:49<03:50, 32.88s/it]

Retry 1/5 failed: Expecting ',' delimiter: line 1 column 84102 (char 84101)


100%|██████████| 34/34 [26:23<00:00, 46.58s/it]

Saved to: /content/drive/MyDrive/data cleaning/stocks_markets_llm_parsed_all.csv
                                   question ticker lower_bound upper_bound
0      Will Apple dip to $268 in Nov 25-28?   AAPL        -268        -268
1      Will Apple dip to $266 in Nov 25-28?   AAPL        -266        -266
2        Will Apple reach $352 in November?   AAPL         352         inf
3        Will Apple reach $332 in November?   AAPL         332         inf
4        Will Apple reach $316 in November?   AAPL         316         inf
5        Will Apple reach $300 in November?   AAPL         300         inf
6        Will Apple reach $288 in November?   AAPL         288         inf
7        Will Apple reach $280 in November?   AAPL         280         inf
8        Will Apple reach $272 in November?   AAPL         272         inf
9       Will Apple dip to $264 in November?   AAPL        -264        -264
10      Will Apple dip to $256 in November?   AAPL        -256        -256
11      Will Apple 

加一个checkpoint以后运行，以防止模型连接时timeout

In [ ]:
import pandas as pd
import requests
import json
import time
import os
from tqdm import tqdm

# =========================
# Config
# =========================
OPENROUTER_API_KEY = "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"
MODEL = "openai/gpt-4o-mini"

base_path = "/content/drive/MyDrive/data cleaning"
input_path = f"{base_path}/stocks_markets_matched.csv"
output_path = f"{base_path}/stocks_markets_llm_parsed.csv"
checkpoint_path = f"{base_path}/stocks_markets_llm_checkpoint.jsonl"

BATCH_SIZE = 25
SLEEP_BETWEEN_CALLS = 3
MAX_RETRIES = 10

# 如果你只想先试500行，就改成 500
# 如果跑全部，就设为 None
N_ROWS = None

# =========================
# Load data
# =========================
df = pd.read_csv(input_path)

if "question" in df.columns:
    question_col = "question"
else:
    question_col = df.columns[1]

if N_ROWS is not None:
    df = df.head(N_ROWS).copy()

df = df.reset_index(drop=False).rename(columns={"index": "row_id"})

print("Using question column:", question_col)
print("Total rows to process:", len(df))


# =========================
# Helpers
# =========================
def chunk_list(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]


def build_messages(batch_records):
    system_prompt = """
You are an information extraction system.

For each input row, extract:
1. ticker
2. lower_bound
3. upper_bound

Return ONLY valid JSON in this exact format:
{"results":[
  {
    "row_id": integer,
    "ticker": string or null,
    "lower_bound": number or "inf" or "-inf" or null,
    "upper_bound": number or "inf" or "-inf" or null
  }
]}

Rules:
- "reach $316", "hit $316", "touch $316", "above $316", "over $316"，">316"
  => lower_bound = 316, upper_bound = "inf"
- "below $316", "under $316"，"dip to $316", "drop to $316", "fall to $316",
  => lower_bound = "-inf", upper_bound = 316
- "$100-$105" or "$100 to $105"
  => lower_bound = 100, upper_bound = 105
- exact bucket like "close at $100"
  => lower_bound = 100, upper_bound = 100
- Do not use date numbers as prices.
- Ignore years, dates, and week ranges unless they are clearly part of a price expression.
- If ticker appears in parentheses, use it.
- If ticker is not in parentheses but the company is obvious, infer the standard stock ticker.
- If not confident, return null for ticker.
- Return exactly one object per input row.

Examples:
"Will Apple reach $316 in November?" -> {"row_id":1,"ticker":"AAPL","lower_bound":316,"upper_bound":"inf"}
"Will Microsoft (MSFT) finish week of November 17 above $490?" -> {"row_id":2,"ticker":"MSFT","lower_bound":490,"upper_bound":"inf"}
"Will Netflix (NFLX) close at $100-$105 in 2025?" -> {"row_id":3,"ticker":"NFLX","lower_bound":100,"upper_bound":105}
"Will Apple dip to $268 in Nov 25-28?" -> {"row_id":4,"ticker":"AAPL","lower_bound":"-inf","upper_bound":"268"}
""".strip()

    user_prompt = "Extract fields for these rows:\n" + json.dumps(batch_records, ensure_ascii=False)

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_openrouter(batch_records):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": MODEL,
        "messages": build_messages(batch_records),
        "temperature": 0,
        "response_format": {"type": "json_object"},
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=180)

            # 专门处理 429
            if response.status_code == 429:
                wait_time = min(120, (2 ** attempt) * 5)
                print(f"429 rate limited. Sleeping {wait_time}s before retry {attempt+1}/{MAX_RETRIES}...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

            data = response.json()
            content = data["choices"][0]["message"]["content"]
            parsed = json.loads(content)

            if "results" not in parsed or not isinstance(parsed["results"], list):
                raise ValueError("Model output missing results array")

            return parsed["results"]

        except Exception as e:
            wait_time = min(120, (2 ** attempt) * 3)
            print(f"Retry {attempt+1}/{MAX_RETRIES} failed: {e}")
            print(f"Sleeping {wait_time}s...")
            time.sleep(wait_time)

    raise RuntimeError("OpenRouter call failed after retries")


def load_checkpoint(checkpoint_path):
    if not os.path.exists(checkpoint_path):
        return {}

    done = {}
    with open(checkpoint_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                done[obj["row_id"]] = obj
    return done


def append_checkpoint(results, checkpoint_path):
    with open(checkpoint_path, "a", encoding="utf-8") as f:
        for obj in results:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


# =========================
# Resume from checkpoint
# =========================
done_map = load_checkpoint(checkpoint_path)
done_row_ids = set(done_map.keys())

print("Already completed rows from checkpoint:", len(done_row_ids))

records = (
    df[["row_id", question_col]]
    .rename(columns={question_col: "question"})
    .to_dict("records")
)

remaining_records = [r for r in records if r["row_id"] not in done_row_ids]
print("Remaining rows:", len(remaining_records))


# =========================
# Main loop
# =========================
for batch in tqdm(list(chunk_list(remaining_records, BATCH_SIZE))):
    batch_results = call_openrouter(batch)
    append_checkpoint(batch_results, checkpoint_path)
    time.sleep(SLEEP_BETWEEN_CALLS)


# =========================
# Rebuild final output from checkpoint
# =========================
done_map = load_checkpoint(checkpoint_path)
parsed_df = pd.DataFrame(list(done_map.values()))

for col in ["row_id", "ticker", "lower_bound", "upper_bound"]:
    if col not in parsed_df.columns:
        parsed_df[col] = None

parsed_df = parsed_df.drop_duplicates(subset=["row_id"], keep="last")

final_df = df.merge(
    parsed_df[["row_id", "ticker", "lower_bound", "upper_bound"]],
    on="row_id",
    how="left"
).drop(columns=["row_id"])

final_df.to_csv(output_path, index=False)

print("Saved final output to:", output_path)
print("Rows with both bounds missing:", ((final_df["lower_bound"].isna()) & (final_df["upper_bound"].isna())).sum())
print(final_df[[question_col, "ticker", "lower_bound", "upper_bound"]].head(20))

Using question column: question
Total rows to process: 6678
Already completed rows from checkpoint: 0
Remaining rows: 6678


100%|██████████| 268/268 [48:28<00:00, 10.85s/it]

Saved final output to: /content/drive/MyDrive/data cleaning/stocks_markets_llm_parsed.csv
Rows with both bounds missing: 0
                                   question ticker lower_bound upper_bound
0      Will Apple dip to $268 in Nov 25-28?   AAPL        -inf         268
1      Will Apple dip to $266 in Nov 25-28?   AAPL        -inf         266
2        Will Apple reach $352 in November?   AAPL         352         inf
3        Will Apple reach $332 in November?   AAPL         332         inf
4        Will Apple reach $316 in November?   AAPL         316         inf
5        Will Apple reach $300 in November?   AAPL         300         inf
6        Will Apple reach $288 in November?   AAPL         288         inf
7        Will Apple reach $280 in November?   AAPL         280         inf
8        Will Apple reach $272 in November?   AAPL         272         inf
9       Will Apple dip to $264 in November?   AAPL        -inf         264
10      Will Apple dip to $256 in November?   AAPL  